# Anthropic API Examples

Companion notebook for [`README.md`](./README.md).  
All examples use the `anthropic` library and load credentials from a `.env` file.

**Requirements:** `conda activate agents` and `pip install anthropic python-dotenv`

In [1]:
# Shared setup — run this cell first
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()          # reads ANTHROPIC_API_KEY from .env
client = Anthropic()   # picks up the key automatically
print("Client ready.")

Client ready.


---
## 1. Basic Message

Every request requires `model`, `max_tokens`, and `messages`.  
The `system` prompt is a **top-level parameter** — not a message role.

In [2]:
message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    system="You are a concise assistant.",
    messages=[
        {"role": "user", "content": "What is the capital of Japan?"},
    ],
)

print(message.content[0].text)
print(f"\nStop reason:    {message.stop_reason}")
print(f"Input tokens:   {message.usage.input_tokens}")
print(f"Output tokens:  {message.usage.output_tokens}")

Tokyo is the capital of Japan.

Stop reason:    end_turn
Input tokens:   21
Output tokens:  10


---
## 2. Multi-Turn Conversation

Maintain context by appending each assistant reply to `messages`.  
Roles must alternate: `user` → `assistant` → `user` → …

In [3]:
messages = []

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        system="You are a helpful assistant.",
        messages=messages,
    )
    reply = response.content[0].text
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("My name is Alice."))
print()
print(chat("What is my name?"))   # model should remember

Nice to meet you, Alice! How can I help you today?

Your name is Alice. You told me that at the beginning of our conversation.


---
## 3. Streaming

### 3a. `.stream()` context manager

Recommended approach — provides a `text_stream` iterator and access to the final message with usage stats.

In [4]:
with client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Write a haiku about Python."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

final = stream.get_final_message()
print(f"\nInput tokens:  {final.usage.input_tokens}")
print(f"Output tokens: {final.usage.output_tokens}")

Serpent of code flows,
Indented lines speak wisdom—
Simple, yet powerful.

Input tokens:  14
Output tokens: 24


### 3b. Low-level `stream=True`

Iterates raw server-sent events. Useful when you need full control over event types.

In [5]:
with client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "List three benefits of Python."}],
    stream=True,
) as stream:
    for event in stream:
        if event.type == "content_block_delta":
            print(event.delta.text, end="", flush=True)
print()

# Three Benefits of Python

1. **Easy to Learn and Read**
   - Simple, clean syntax that resembles natural language
   - Lower barrier to entry for beginners
   - Faster development and easier code maintenance

2. **Versatile and Widely Used**
   - Applicable across many domains: web development, data science, artificial intelligence, automation, and more
   - Large ecosystem of libraries and frameworks (Django, NumPy, TensorFlow, etc.)
   - Strong community support and abundant learning resources

3. **High Productivity**
   - Requires fewer lines of code to accomplish tasks compared to other languages
   - Built-in data structures and functions reduce development time
   - Ideal for rapid prototyping and quick problem-solving


---
## 4. Tool Use / Function Calling

1. Define tools with `input_schema`.  
2. First call: model returns `stop_reason = "tool_use"`.  
3. Execute the function locally.  
4. Return a `tool_result` message to get the final answer.

In [6]:
import json

tools = [
    {
        "name": "get_weather",
        "description": "Return current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    }
]

messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]

# First call
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    tools=tools,
    messages=messages,
)
print("Stop reason:", response.stop_reason)

if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    print(f"Tool called: {tool_block.name} | {tool_block.input}")

    # Simulate the actual function
    tool_result = {"temperature": 18, "condition": "Cloudy", "city": tool_block.input["city"]}

    messages.append({"role": "assistant", "content": response.content})
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": json.dumps(tool_result),
        }],
    })

    # Second call — final answer
    final = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        tools=tools,
        messages=messages,
    )
    print("\nFinal answer:", final.content[0].text)

Stop reason: tool_use
Tool called: get_weather | {'city': 'Tokyo'}

Final answer: The weather in Tokyo is currently:
- **Temperature:** 18°C (about 64°F)
- **Condition:** Cloudy

It's a mild, cloudy day in Tokyo. You might want to bring an umbrella if you're planning to head out, just in case!


---
## 5. Structured Output with Pydantic

Use `client.messages.parse()` with `output_format` set to a Pydantic model.  
The SDK handles JSON schema generation and parsing automatically.

In [7]:
import pydantic

class CalendarEvent(pydantic.BaseModel):
    title: str
    date: str
    time: str
    attendees: list[str]
    location: str | None = None

parsed = client.messages.parse(
    model="claude-sonnet-4-5",
    max_tokens=512,
    output_format=CalendarEvent,
    messages=[{
        "role": "user",
        "content": "Extract: Team meeting tomorrow at 3pm with Alice and Bob in Room A.",
    }],
)

event = parsed.parsed_output
print(f"Title:     {event.title}")
print(f"Date:      {event.date}")
print(f"Time:      {event.time}")
print(f"Attendees: {', '.join(event.attendees)}")
print(f"Location:  {event.location}")

Title:     Team meeting
Date:      tomorrow
Time:      3pm
Attendees: Alice, Bob
Location:  Room A


---
## 6. Vision (Image Input)

Pass images in `content` alongside text. Supports public URLs and base64 local files.

### 6a. Image from URL

In [8]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "url",
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/400px-2023_06_08_Raccoon1.jpg",
                },
            },
            {"type": "text", "text": "What animal is in this image?"},
        ],
    }],
)
print(response.content[0].text)

The animal in this image is a **raccoon**. You can identify it by its distinctive features, including the characteristic black "mask" across its eyes, its rounded ears, and its grayish-brown fur. The raccoon appears to be perched on or near a tree trunk at night, which is typical behavior as raccoons are primarily nocturnal animals.


### 6b. Image from local file (base64)

In [9]:
import base64

# Replace with a real local image path to test
image_path = "../openai_api/assets/2023_06_08_Raccoon1.jpg"

try:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": b64,
                    },
                },
                {"type": "text", "text": "Describe this image."},
            ],
        }],
    )
    print(response.content[0].text)
except FileNotFoundError:
    print(f"File not found: {image_path} — replace with a real image path to test.")

# Image Description

This is a nighttime photograph of a **raccoon** perched on a tree trunk. The image captures the raccoon's distinctive features:

- **Characteristic mask**: The black facial markings across its eyes are clearly visible
- **Rounded ears**: Visible and alert
- **Light coloring**: The face and fur appear highlighted against the dark background
- **Position**: The raccoon is clinging to the textured bark of a large tree

The photograph uses excellent lighting to illuminate the raccoon's face while keeping the background dark, creating a striking contrast. The textured tree bark fills much of the frame, emphasizing the nocturnal nature of the subject. This is a classic wildlife shot capturing the raccoon in its natural environment during nighttime hours.


---
## 7. Extended Thinking

Enable Claude to reason internally before answering by setting `thinking.type = "enabled"` and allocating a `budget_tokens` for internal reasoning.  
The response includes a `thinking` block followed by the final `text` block.

> Requires `claude-sonnet-4-5` or later. `max_tokens` must exceed `budget_tokens`.

In [10]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=8000,
    thinking={
        "type": "enabled",
        "budget_tokens": 5000,   # tokens reserved for internal reasoning
    },
    messages=[{
        "role": "user",
        "content": "A train leaves Chicago at 9am at 60 mph. Another leaves New York at 10am at 80 mph toward Chicago (800 miles apart). When and where do they meet?",
    }],
)

for block in response.content:
    if block.type == "thinking":
        print("=== Thinking (first 400 chars) ===")
        print(block.thinking[:400], "...\n")
    elif block.type == "text":
        print("=== Answer ===")
        print(block.text)

=== Thinking (first 400 chars) ===
Let me set up this problem carefully.

- Train 1 leaves Chicago at 9am at 60 mph heading toward New York
- Train 2 leaves New York at 10am at 80 mph heading toward Chicago
- The cities are 800 miles apart

Let me set up coordinates where Chicago is at position 0 and New York is at position 800.

Train 1's position at time t (hours after 9am): 
- Position = 60t

Train 2's position at time t (hours  ...

=== Answer ===
# Train Problem Solution

Let me work through this step-by-step.

**Setting up the problem:**
- Train 1 (Chicago): Leaves at 9am, speed = 60 mph
- Train 2 (New York): Leaves at 10am, speed = 80 mph  
- Distance between cities = 800 miles

**Finding when they meet:**

Let t = hours after 9am

- Train 1 position from Chicago: 60t
- Train 2 position from Chicago: 800 - 80(t-1) [since it starts 1 hour later]

Setting them equal:
- 60t = 800 - 80(t-1)
- 60t = 800 - 80t + 80
- 140t = 880
- t = 880/140 = 44/7 hours ≈ 6.29 hours

**Time they meet

---
## 8. Async Client

Use `AsyncAnthropic` for non-blocking calls in async frameworks (FastAPI, asyncio).  

> In Jupyter, `await` works directly in cells — no `asyncio.run()` needed.

In [11]:
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic()   # also reads ANTHROPIC_API_KEY from env

# Basic async message
response = await async_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Name three planets."}],
)
print(response.content[0].text)

Here are three planets:

1. **Earth**
2. **Mars**
3. **Jupiter**


In [12]:
# Async streaming
async with async_client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Count to five, one word per line."}],
) as stream:
    async for text in stream.text_stream:
        print(text, end="", flush=True)
print()

One
Two
Three
Four
Five


---
## 9. Token Counting

Estimate token usage **before** making a call — useful for context limit checks and cost estimation.

In [13]:
count = client.messages.count_tokens(
    model="claude-haiku-4-5",
    system="You are a helpful assistant.",
    messages=[
        {"role": "user",      "content": "Hello, how are you?"},
        {"role": "assistant", "content": "I'm doing well, thank you!"},
        {"role": "user",      "content": "Can you explain quantum entanglement?"},
    ],
)

print(f"Estimated input tokens: {count.input_tokens}")

Estimated input tokens: 41


---
## 10. Files API

Upload files once and reference them by `file_id` across many requests — avoids re-uploading large PDFs or images.

> **Beta:** use `client.beta.files.*` and pass `betas=["files-api-2025-04-14"]` to message calls.

Supported types: `application/pdf` → `document` block; `image/*` → `image` block; `text/plain` → `document` block.  
Limits: 500 MB per file, 500 GB total per organisation.

In [14]:
import tempfile, os

# Create a small plain-text file to upload (no real PDF needed to test)
tmp = tempfile.NamedTemporaryFile(suffix=".txt", delete=False, mode="w")
tmp.write("This is a sample document.\nIt has two lines.")
tmp.flush()
tmp_path = tmp.name
tmp.close()

# Upload
with open(tmp_path, "rb") as f:
    uploaded = client.beta.files.upload(
        file=("sample.txt", f, "text/plain"),
    )

file_id = uploaded.id
print(f"Uploaded file_id: {file_id}")
os.unlink(tmp_path)

Uploaded file_id: file_011CZckAN8gS8a1AHu5wAHS9


In [15]:
# Use the uploaded file in a message (document block for text/PDF)
response = client.beta.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Summarise this document in one sentence."},
            {
                "type": "document",
                "source": {"type": "file", "file_id": file_id},
            },
        ],
    }],
    betas=["files-api-2025-04-14"],
)
print(response.content[0].text)

This is a brief two-line sample document.


In [16]:
# List uploaded files
files = client.beta.files.list()
for f in files.data:
    print(f.id, getattr(f, "filename", "—"))

# Get metadata for our file
meta = client.beta.files.retrieve_metadata(file_id)
print(f"\nMetadata: {meta}")

# Delete the file
client.beta.files.delete(file_id)
print(f"\nDeleted {file_id}")

file_011CZckAN8gS8a1AHu5wAHS9 sample.txt

Metadata: FileMetadata(id='file_011CZckAN8gS8a1AHu5wAHS9', created_at=datetime.datetime(2026, 4, 1, 8, 35, 27, 626000, tzinfo=datetime.timezone.utc), filename='sample.txt', mime_type='text/plain', size_bytes=45, type='file', downloadable=False)

Deleted file_011CZckAN8gS8a1AHu5wAHS9


---
## 11. Agent SDK

The SDK ships beta utilities for building agentic loops without boilerplate:

- **`@beta_tool`** — auto-generates `input_schema` from Python function signature + docstring.
- **`tool_runner()`** — drives the full tool-calling loop automatically; yields a `BetaMessage` per round-trip.
- **`ToolError`** — raise inside a tool to return structured errors (text, images) the model can reason about.

### 11a. `@beta_tool` — define tools from Python functions

In [17]:
from anthropic import beta_tool

@beta_tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Return the current weather for a city.

    Args:
        city: The city name.
        unit: Temperature unit, either 'celsius' or 'fahrenheit'.
    """
    return f"18°{'C' if unit == 'celsius' else 'F'}, Cloudy in {city}"

@beta_tool
def add(left: int, right: int) -> str:
    """Add two integers.

    Args:
        left: First integer.
        right: Second integer.
    """
    return str(left + right)

# Inspect the auto-generated schema
import json
print(json.dumps(get_weather.to_dict(), indent=2))

{
  "name": "get_weather",
  "description": "Return the current weather for a city.",
  "input_schema": {
    "additionalProperties": false,
    "properties": {
      "city": {
        "description": "The city name.",
        "title": "City",
        "type": "string"
      },
      "unit": {
        "default": "celsius",
        "description": "Temperature unit, either 'celsius' or 'fahrenheit'.",
        "title": "Unit",
        "type": "string"
      }
    },
    "required": [
      "city"
    ],
    "type": "object"
  }
}


### 11b. `tool_runner()` — automatic agentic loop

Drives the full tool-calling loop: calls tools, feeds results back, stops when `stop_reason == "end_turn"`.  
Yields one `BetaMessage` per round-trip so you can observe each step.

In [19]:
runner = client.beta.messages.tool_runner(
    model="claude-haiku-4-5",
    max_tokens=512,
    tools=[get_weather, add],
    messages=[{"role": "user", "content": "What is 9 + 10, and what's the weather in Tokyo?"}],
)

for i, message in enumerate(runner):
    print(f"--- Round {i+1} (stop_reason: {message.stop_reason}) ---")
    for block in message.content:
        if hasattr(block, "text"):
            print("TEXT:", block.text)
        elif block.type == "tool_use":
            print(f"TOOL CALL: {block.name}({block.input})")
        elif block.type == "tool_result":
            print(f"TOOL RESULT: {block.content}")

--- Round 1 (stop_reason: tool_use) ---
TOOL CALL: add({'left': 9, 'right': 10})
TOOL CALL: get_weather({'city': 'Tokyo'})
--- Round 2 (stop_reason: end_turn) ---
TEXT: Great! Here are the answers:

1. **9 + 10 = 19**

2. **Weather in Tokyo**: It's currently 18°C (about 64°F) and cloudy.


### 11c. `ToolError` — structured error feedback

Raise `ToolError` inside a `@beta_tool` to return structured content (text, images) that the model can reason about instead of a bare exception string.

In [20]:
from anthropic.lib.tools import ToolError

@beta_tool
def fetch_url(url: str) -> str:
    """Fetch the content of a URL.

    Args:
        url: The URL to fetch (must be HTTPS).
    """
    if not url.startswith("https://"):
        raise ToolError(f"Only HTTPS URLs are supported. Got: {url}")
    return f"<html>content of {url}</html>"

# The model will receive the ToolError message and explain what went wrong
runner = client.beta.messages.tool_runner(
    model="claude-haiku-4-5",
    max_tokens=512,
    tools=[fetch_url],
    messages=[{"role": "user", "content": "Fetch http://example.com for me."}],
)

for message in runner:
    for block in message.content:
        if hasattr(block, "text"):
            print(block.text)

I appreciate your request, but I need to clarify that the fetch_url function I have access to only works with HTTPS URLs (secure connections), not HTTP URLs.

The URL you've provided is `http://example.com`, which uses the non-secure HTTP protocol. To fetch this content, I would need an HTTPS version of the URL.

Could you either:
1. Provide an HTTPS version of the URL (e.g., `https://example.com`), or
2. Confirm if you'd like me to try fetching `https://example.com` instead?


### 11d. Manual agent loop

For full control — equivalent to what `tool_runner()` does internally.

In [21]:
tools = [add, get_weather]
messages = [{"role": "user", "content": "What is 42 + 58, and what's the weather in Paris?"}]

while True:
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=512,
        tools=[t.to_dict() for t in tools],
        messages=messages,
    )

    messages.append({"role": "assistant", "content": response.content})

    if response.stop_reason == "end_turn":
        print("Final answer:", response.content[0].text)
        break

    # Execute all tool calls in this round
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            tool_fn = next(t for t in tools if t.name == block.name)
            result = tool_fn(**block.input)
            print(f"  Executed {block.name}({block.input}) → {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result,
            })

    messages.append({"role": "user", "content": tool_results})

  Executed add({'left': 42, 'right': 58}) → 100
  Executed get_weather({'city': 'Paris'}) → 18°C, Cloudy in Paris
Final answer: 42 + 58 = **100**

The weather in Paris is currently **18°C and Cloudy**.
